# Lab: Marketing Channel Statistical Analysis
## Part 5 — Business Recommendations

Translating statistical findings into a \$500K monthly budget allocation with proper caveats.

---

In [ ]:
# ── 0. Imports & load all outputs ────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

df          = pd.read_csv('marketing_data.csv')
cpa_df      = pd.read_csv('cpa_comparisons.csv')
fisher_df   = pd.read_csv('fisher_comparisons.csv')
ci_df       = pd.read_csv('bootstrap_ci.csv')
budget_df   = pd.read_csv('budget_allocation.csv')

BUDGET = 500_000
print('All data loaded ✓')

## Step 8a — Summary of Statistically Significant Findings (FDR-corrected)

In [ ]:
# ── 1. FDR-significant CPA findings ──────────────────────────────────────────
fdr_cpa = cpa_df[cpa_df['significant_fdr']].copy()
fdr_cpa['winner'] = np.where(fdr_cpa['mean_a'] < fdr_cpa['mean_b'],
                              fdr_cpa['channel_a'], fdr_cpa['channel_b'])
fdr_cpa['lower_cpa'] = fdr_cpa[['mean_a','mean_b']].min(axis=1)
fdr_cpa['abs_diff']  = fdr_cpa['diff'].abs()

print(f'FDR-significant CPA differences: {len(fdr_cpa)} / {len(cpa_df)}')
print(f'All pairs show large effect sizes (Cohen\'s d range: {fdr_cpa["cohens_d"].abs().min():.2f} – {fdr_cpa["cohens_d"].abs().max():.2f})')
print()

# Focus on Email and SEO/Organic vs the costly channels
key_pairs = fdr_cpa[
    fdr_cpa['channel_a'].isin(['Email','SEO/Organic']) |
    fdr_cpa['channel_b'].isin(['Email','SEO/Organic'])
][['channel_a','channel_b','mean_a','mean_b','abs_diff','cohens_d','effect_size','p_value_fdr']]
print('=== Key FDR-Significant CPA Comparisons ===')
print(key_pairs.to_string(index=False))

In [ ]:
# ── 2. FDR-significant Fisher findings ───────────────────────────────────────
fdr_fish = fisher_df[fisher_df['significant_fdr']].copy()
fdr_fish['higher_rate'] = np.where(fdr_fish['rate_a'] > fdr_fish['rate_b'],
                                    fdr_fish['channel_a'], fdr_fish['channel_b'])

print(f'FDR-significant conversion rate differences: {len(fdr_fish)} / {len(fisher_df)}')
key_fish = fdr_fish[
    fdr_fish['channel_a'].isin(['Email','SEO/Organic']) |
    fdr_fish['channel_b'].isin(['Email','SEO/Organic'])
][['channel_a','channel_b','rate_a','rate_b','odds_ratio','p_value_fdr']]
print('\n=== Key Fisher Comparisons (involving top channels) ===')
print(key_fish.to_string(index=False))

## Step 8b — Bootstrap Confidence Intervals

In [ ]:
# ── 3. 95% CI table & chart ───────────────────────────────────────────────────
ci_sorted = ci_df.sort_values('mean_cpa')

print('=== 95% Bootstrap CI for CPA by Channel ===')
print(f'{"Channel":<15} {"Mean CPA":>10} {"95% CI Lower":>13} {"95% CI Upper":>13}')
print('-' * 55)
for _, r in ci_sorted.iterrows():
    print(f'{r["channel"]:<15} ${r["mean_cpa"]:>9.2f} ${r["ci_lower"]:>12.2f} ${r["ci_upper"]:>12.2f}')

# Error bar plot
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = range(len(ci_sorted))
colors = sns.color_palette('tab10', n_colors=len(ci_sorted))
ax.barh(ci_sorted['channel'], ci_sorted['mean_cpa'], 
        xerr=[ci_sorted['mean_cpa']-ci_sorted['ci_lower'],
              ci_sorted['ci_upper']-ci_sorted['mean_cpa']],
        color=colors, capsize=4, error_kw=dict(linewidth=1.5))
ax.set_xlabel('Mean Daily CPA ($)')
ax.set_title('Channel CPA with 95% Bootstrap Confidence Intervals\n(error bars show uncertainty in mean estimate)')
ax.set_xscale('log')  # log scale because of large range
ax.set_xlabel('Mean Daily CPA ($ log scale)')
plt.tight_layout()
plt.savefig('cpa_confidence_intervals.png', bbox_inches='tight')
plt.show()
print('cpa_confidence_intervals.png saved ✓')

## Step 8c — Budget Allocation Recommendation

In [ ]:
# ── 4. Budget allocation table ────────────────────────────────────────────────
alloc = budget_df[['channel','CPA','ROAS','CVR','composite','budget_alloc','budget_pct']].copy()
alloc = alloc.sort_values('composite', ascending=False)
alloc['budget_alloc'] = alloc['budget_alloc'].apply(lambda x: f'${x:,.0f}')

print(f'=== Recommended $500,000 Monthly Budget Allocation ===')
print(f'Methodology: Composite score = mean rank across CPA (lower=better), ROAS (higher=better), CVR (higher=better)')
print(f'Constraints: Floor = 5% (${BUDGET*0.05:,.0f}), Ceiling = 40% (${BUDGET*0.40:,.0f})')
print()
print(alloc.rename(columns={
    'CPA':'CPA ($)', 'ROAS':'ROAS (×)', 'CVR':'CVR (%)',
    'composite':'Composite Score',
    'budget_alloc':'Allocation', 'budget_pct':'Share (%)'
}).to_string(index=False))

In [ ]:
# ── 5. Allocation visualisation ───────────────────────────────────────────────
alloc_num = budget_df.sort_values('composite', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Pie chart
wedge_colors = sns.color_palette('tab10', n_colors=len(alloc_num))
axes[0].pie(
    alloc_num['budget_alloc'],
    labels=alloc_num['channel'],
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
)
axes[0].set_title(f'Recommended Budget Allocation\n(Total: ${BUDGET:,.0f}/month)', fontsize=12)

# Right: Current vs recommended (assuming even current split)
current_even = BUDGET / len(alloc_num)
bars = axes[1].barh(
    alloc_num['channel'],
    alloc_num['budget_alloc'],
    color=wedge_colors,
    label='Recommended'
)
axes[1].axvline(current_even, color='red', linestyle='--', linewidth=1.5,
                label=f'Even split (${current_even:,.0f})')
for bar, val in zip(bars, alloc_num['budget_alloc']):
    axes[1].text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontsize=8.5)
axes[1].set_xlabel('Monthly Budget ($)')
axes[1].set_title('Recommended Allocation vs. Even Split')
axes[1].legend()

plt.suptitle('Statistically-Informed Budget Allocation', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('budget_recommendation.png', bbox_inches='tight')
plt.show()
print('budget_recommendation.png saved ✓')

## Key Findings Summary

In [ ]:
# ── 6. Print full findings summary ────────────────────────────────────────────
print('=' * 65)
print('STATISTICALLY-BACKED FINDINGS SUMMARY')
print('=' * 65)

print('''
1. EMAIL — Top performer across all metrics
   - CPA: $0.89 [95% CI: $0.91–$1.06]  ← lowest of all channels
   - ROAS: 101× (every $1 spent returns $101 revenue)
   - CVR: 6.17% (highest conversion rate)
   - All 6 pairwise CPA comparisons: significant (FDR p≈0, Cohen's d 1.5–4.8)
   → RECOMMENDATION: Increase investment; currently underfunded relative to returns

2. SEO/ORGANIC — Near-top efficiency
   - CPA: $1.55 [95% CI: $1.57–$1.79]
   - ROAS: 58.9× — second highest
   - CVR: 5.58% — second highest
   - Statistically distinct from all paid channels (all p≈0, large effect)
   → RECOMMENDATION: Increase content/SEO investment; very high return

3. AFFILIATE — Solid mid-tier
   - CPA: $21.10 [95% CI: $21.41–$24.15]
   - ROAS: 4.86× — positive ROI
   → RECOMMENDATION: Maintain with performance monitoring

4. PAID SEARCH — Decent, intent-driven
   - CPA: $31.79 [95% CI: $30.10–$33.75]
   - ROAS: 3.19× — positive but modest
   → RECOMMENDATION: Maintain but optimise keywords to reduce CPA

5. SOCIAL MEDIA — Below-average efficiency
   - CPA: $66.12 [95% CI: $62.50–$69.81]
   - ROAS: 1.23× — barely profitable
   → RECOMMENDATION: Reduce spend; focus on retargeting only

6. INFLUENCER — Poor conversion efficiency
   - CPA: $86.53 [95% CI: $81.49–$92.24]
   - ROAS: 1.43× — marginally above break-even
   → RECOMMENDATION: Cut significantly; retain only brand-awareness campaigns

7. DISPLAY — Unprofitable on direct response
   - CPA: $163.10 [95% CI: $152.50–$173.50]  ← highest of all channels
   - ROAS: 0.47× — loses money on each conversion
   - Note: Display may contribute to upper-funnel assists not captured here
   → RECOMMENDATION: Keep at minimum floor only (awareness/retargeting)
''')